### setup

In [6]:
import os
import torch
import json 
from unsloth import FastLanguageModel
from trl import SFTTrainer , SFTConfig 
from datasets import Dataset

os.environ["HF_HUB_DISABLE_XET"] = "1"

os.environ["UNSLOTH_DISABLE_FUSED_XET"] = "1"

print("library loaded")
print(f"torch version:{torch.__version__}")
print(f"cuda version:{torch.cuda.is_available()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\HP\anaconda3\envs\unsloth-ft\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0626 02:56:06.460000 15672 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0626 02:56:06.652000 15672 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


library loaded
torch version:2.12.0+cu130
cuda version:True


### LOAD MODLE 

In [7]:
model_path = r"c:\models\gemma2"

print("modle loaded... ", model_path) 

model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path ,
    device_map="auto",
    dtype = None,
    max_seq_length = 2048 , 
    load_in_4bit= True , 
)

print("modle loaded... ")
print(f"device:{model.device}")
print(f"dtype:{model.dtype}")

modle loaded...  c:\models\gemma2
==((====))==  Unsloth 2026.6.1: Fast Gemma patching. Transformers: 5.11.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 164/164 [00:07<00:00, 20.80it/s]
Unsloth: Will load c:\models\gemma2 as a legacy tokenizer.


modle loaded... 
device:cuda:0
dtype:torch.bfloat16


### LOAD MODEL FOR (LORA) PART 

In [8]:
print(":arrows_counterclockwise:get ready for LORA...")

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print(":white_check_mark:LORA is ready")
print(f":white_check_mark:ready parameters for training...: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}")

:arrows_counterclockwise:get ready for LORA...


Unsloth 2026.6.1 patched 18 layers with 18 QKV layers, 18 O layers and 18 MLP layers.


:white_check_mark:LORA is ready
:white_check_mark:ready parameters for training...: 19.61


### LOAD DATE FROM JSON

In [9]:
dataset_path = "data.json" 

print(f":arrows_counterclockwise: load data from json : {dataset_path}")

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f":white_check_mark: load {len(data)} samples from json file :{len(data)}")

print("\n:memo:data sample")
print(json.dumps(data[0], indent=2, ensure_ascii=False)[:500] + "...")

:arrows_counterclockwise: load data from json : data.json
:white_check_mark: load 393 samples from json file :393

:memo:data sample
{
  "messages": [
    {
      "role": "system",
      "content": "تو یه دستیار مفید، مهربون، دوستانه و صمیمی هستی. همیشه با لحن گرم، روشن و همراه جواب می‌دی."
    },
    {
      "role": "user",
      "content": "سلام!"
    },
    {
      "role": "assistant",
      "content": "سلام عزیزم، خوش اومدی! چطور می‌تونم کمکت کنم؟"
    }
  ]
}...


In [10]:
def format_conversation(example):
    if isinstance(example, list):
        text = ""
        for msg in example:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            if role == "user":
                text += f"<|user|>\n{content}\n"
            elif role == "assistant":
                text += f"<|assistant|>\n{content}\n"
            elif role == "system":
                text += f"<|system|>\n{content}\n"
        return {"text": text}
    
    elif isinstance(example, dict) and "messages" in example:
        text = ""
        for msg in example["messages"]:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            if role == "user":
                text += f"<|user|>\n{content}\n"
            elif role == "assistant":
                text += f"<|assistant|>\n{content}\n"
            elif role == "system":
                text += f"<|system|>\n{content}\n"
        return {"text": text}
    
    elif isinstance(example, dict) and "user" in example and "assistant" in example:
        text = f"<|user|>\n{example['user']}\n<|assistant|>\n{example['assistant']}\n"
        return {"text": text}
    
    else:
        raise ValueError("format is not found pleas send again")

print(":arrows_counterclockwise:transform the data...")
formatted_data = [format_conversation(item) for item in data]
dataset = Dataset.from_list(formatted_data)

print(f":white_check_mark: simple axample of the dataset: {len(dataset)} ")
print("\n:memo: first example of the dataset:")
print(dataset[0]["text"][:300] + "...")

:arrows_counterclockwise:transform the data...
:white_check_mark: simple axample of the dataset: 393 

:memo: first example of the dataset:
<|system|>
تو یه دستیار مفید، مهربون، دوستانه و صمیمی هستی. همیشه با لحن گرم، روشن و همراه جواب می‌دی.
<|user|>
سلام!
<|assistant|>
سلام عزیزم، خوش اومدی! چطور می‌تونم کمکت کنم؟
...


### START TRAINAINING MODEL


In [ ]:
print(":arrows_counterclockwise: start processing...")
print(":hourglass_flowing_sand: just take a moment...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,  
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        output_dir = "outputs",
        dataset_text_field = "text",
        max_seq_length = 2048,
        packing = False,
    ),
)

trainer.train()
print(":white_check_mark: train was successful!")

### SAVE THE TRAINED MODLE 

In [ ]:
print("\n:rocket: saving the model")
model.save_pretrained("gemma_persian_v1")
tokenizer.save_pretrained("gemma_persian_v1")

print("the models saved in the gemma_persian_v1")
print("save directory :", os.path.abspath("gemma_persian_v1"))

print("\n saved files :")
for file in os.listdir("gemma_persian_v1"):
    size =  os.path.getsize(os.path.join("gemma_persian_v1",file))

print(f" - {file} ({size:1f}) KB")


### TESTING TRAINED MODEL 

In [ ]:
print(":arrows_counterclockwise: load model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "gemma_persian_v1",
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "auto",
)

print(":white_check_mark: load model was seccessfully")

### TESTING MODEL 

In [ ]:
def ask_model(prompt, max_tokens=200, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(
        **inputs,
        max_new_tokens = max_tokens,
        temperature = temperature,
        do_sample = True,
        pad_token_id = tokenizer.eos_token_id,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

print("🧪 testing model...\n")
print("-" * 50)

prompt1 = "سلام! چطوری؟"
print(f":memo: سوال: {prompt1}")
print(f"پاسخ: {ask_model(prompt1)}\n")
print("-" * 50)

prompt2 = "به نظرت هوش مصنوعی چیه؟"
print(f":memo: سوال: {prompt2}")
print(f"پاسخ: {ask_model(prompt2)}\n")
print("-" * 50)

### THIS METOD FOR ORGANIZE (ONLY PERSIAN LANGUAGE)

In [ ]:
import unicodedata
import re

def clean_persian_text(text):
    """
    این تابع متن خروجی مدل رو برای نمایش درست فارسی تمیز میکنه.
    """
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('ي', 'ی')
    text = text.replace('ك', 'ک')
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    return text.strip()

sample = "سلام .  چطوری ؟  این یه   متن   فارسیه"
print(":white_check_mark: نمونه قبل:", sample)
print(":white_check_mark: نمونه بعد:", clean_persian_text(sample))

In [ ]:
def ask_model_clean(prompt, max_tokens=300, temperature=0.6):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(
        **inputs,
        max_new_tokens = max_tokens,
        temperature = temperature,
        do_sample = True,
        pad_token_id = tokenizer.eos_token_id,
    )
    raw_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    clean_response = clean_persian_text(raw_response)
    return clean_response

print(":white_check_mark: new function ready for use :")

### FAINAL TEST 

In [ ]:
print("🧪 شروع تست با خروجی تمیز فارسی...\n")
print("=" * 60)

questions = [
    "سلام! چطوری؟",
    "میشه کمکم کنی؟",
]

for q in questions:
    print(f":memo: سوال: {q}")
    answer = ask_model_clean(q)
    print(f"🤖 پاسخ: {answer}")
    print("-" * 60)
    print()
print("🧪 تست تمام شد :white check_mark:\n")